# Mapping National Road Coverage Across Mozambique's Districts

**A geospatial analysis quantifying how much of Mozambique's national road network (N-roads) passes through each administrative district.**

## Why this matters
National roads (N-roads) are the backbone of a country's transport network, connecting provincial capitals, ports, and rural regions. Understanding how road infrastructure is distributed across districts is a first step in transport planning, rural accessibility assessments, and infrastructure investment prioritization.

## Data
- **Roads:** OpenStreetMap road network for Mozambique (`gis_osm_roads_free`), filtered to national roads (`ref` starting with `N`).
- **Administrative boundaries:** District-level (`adm2`) boundary polygons for Mozambique.

## Method
1. Load roads and district boundaries from GeoPackage layers.
2. Filter to national roads using the `ref` attribute.
3. Reproject to a metric, locally-appropriate projected CRS (**EPSG:32736 — UTM Zone 36S**) so length/area calculations are in real meters, not degrees.
4. Compute road length per segment and district area.
5. Spatially join roads to the district they fall within.
6. Aggregate total national road length per district.
7. Visualize and export results.


## 1. Load libraries and data

In [1]:
import geopandas as gpd 
from pathlib import Path

In [5]:
layers = gpd.list_layers("mozambique.gpkg")
print(layers)

                              name geometry_type
0             gis_osm_traffic_free         Point
1              gis_osm_places_free         Point
2           gis_osm_transport_free         Point
3                gis_osm_pois_free         Point
4             gis_osm_natural_free         Point
5                gis_osm_pofw_free         Point
6           gis_osm_waterways_free    LineString
7               gis_osm_roads_free    LineString
8             gis_osm_water_a_free  MultiPolygon
9            gis_osm_places_a_free  MultiPolygon
10          gis_osm_landuse_a_free  MultiPolygon
11          gis_osm_natural_a_free  MultiPolygon
12             gis_osm_pois_a_free  MultiPolygon
13       gis_osm_adminareas_a_free  MultiPolygon
14           gis_osm_railways_free    LineString
15        gis_osm_buildings_a_free  MultiPolygon
16        gis_osm_transport_a_free  MultiPolygon
17          gis_osm_traffic_a_free  MultiPolygon
18  gis_osm_protected_areas_a_free  MultiPolygon
19             gis_o

In [6]:
road = gpd.read_file("mozambique.gpkg", layer="gis_osm_roads_free")
boundary = gpd.read_file("boundaries.gpkg")

In [8]:
road

,osm_id,code,fclass,name,ref,oneway,maxspeed,layer,bridge,tunnel,geometry
0,4360400,5121,unclassified,Av. Ho Chi Min,1080,F,0,0,F,F,"LINESTRING (32.57986 -25.97123, 32.57964 -25.9..."
1,4360401,5122,residential,Rua D. Almeida Ribeiro,1057,F,60,0,F,F,"LINESTRING (32.5817 -25.97331, 32.58222 -25.97..."
2,4360402,5122,residential,Rua John Issa,1089,B,0,0,F,F,"LINESTRING (32.57692 -25.97147, 32.57791 -25.9..."
3,4360403,5114,secondary,Av. Patrice Lumumba,1064,F,0,0,F,F,"LINESTRING (32.5758 -25.97084, 32.57692 -25.97..."
4,4360404,5122,residential,Rua José Sidumo,1059,F,0,0,F,F,"LINESTRING (32.5819 -25.97056, 32.58182 -25.97..."
...,...,...,...,...,...,...,...,...,...,...,...
588112,1552107684,5142,track,,,B,0,0,F,F,"LINESTRING (32.81909 -21.30042, 32.8191 -21.30..."
588113,1552107685,5142,track,,,B,0,0,F,F,"LINESTRING (32.80309 -21.29036, 32.80354 -21.2..."
588114,1552107686,5142,track,,,B,0,0,F,F,"LINESTRING (32.79659 -21.28072, 32.7967 -21.28..."
588115,1552107687,5142,track,,,B,0,0,F,F,"LINESTRING (32.80513 -21.29283, 32.80503 -21.2..."


## 2. Filter to national roads (`N`-prefixed `ref`)

In [15]:
n_road = road[road["ref"].str.match("^N")]

## 3. Reproject to a projected CRS for accurate metric measurements

Geographic CRS (lat/lon, e.g. EPSG:4326) cannot give meaningful lengths or areas — degrees aren't a unit of distance. Reprojecting to **UTM Zone 36S (EPSG:32736)**, the correct UTM zone for Mozambique, lets `.length` and `.area` return real metres.

In [17]:
n_road_rep = n_road.to_crs("EPSG:32736") 

In [23]:
n_road_rep.crs

<Projected CRS: EPSG:32736>
Name: WGS 84 / UTM zone 36S
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Between 30°E and 36°E, southern hemisphere between 80°S and equator, onshore and offshore. Burundi. Eswatini (Swaziland). Kenya. Malawi. Mozambique. Rwanda. South Africa. Tanzania. Uganda. Zambia. Zimbabwe.
- bounds: (30.0, -80.0, 36.0, 0.0)
Coordinate Operation:
- name: UTM zone 36S
- method: Transverse Mercator
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

## 4. Compute road segment lengths

In [20]:
n_road_rep["length"] = n_road_rep["geometry"].length
# This returns the length of each in metres
n_road_rep

,osm_id,code,fclass,name,ref,oneway,maxspeed,layer,bridge,tunnel,geometry,length
16,5172990,5112,trunk,Estrada Nacional Nr. 1,N1,F,60,0,F,F,"LINESTRING (456638.57 7137728.546, 456660.89 7...",892.873965
17,6246374,5112,trunk,,N4,B,80,0,F,F,"LINESTRING (398115.505 7185604.714, 398126.789...",244.752803
28,22731893,5113,primary,Avenida Eduardo Mondlane,N221,F,0,0,F,F,"LINESTRING (500367.434 7287621.716, 500358.257...",834.578265
29,22731908,5113,primary,,N102,B,0,0,F,F,"LINESTRING (580288.303 7232680.621, 580236.854...",25024.286373
30,22731919,5112,trunk,,N1,B,0,0,F,F,"LINESTRING (573116.494 7231303.002, 573057.906...",6812.730391
...,...,...,...,...,...,...,...,...,...,...,...,...
586052,1547709291,5113,primary,N2,N2,B,60,0,T,F,"LINESTRING (441301.766 7122986, 441266.895 712...",68.513679
586053,1547709292,5113,primary,N2,N2,B,60,0,F,F,"LINESTRING (438803.281 7120673.406, 438785.132...",376.526779
586054,1547709293,5113,primary,N2,N2,B,60,0,F,F,"LINESTRING (438549.594 7120425.23, 438492.577 ...",816.040646
586838,1548947891,5115,tertiary,Estrada velha da Moamba,N2,B,60,0,F,F,"LINESTRING (444761.143 7143428.179, 444756.433...",526.313596


In [22]:
total_length = n_road_rep["length"].sum()
print(f"The total length of national road {total_length/1000:.2f} KM.")

The total length of national road 11052.62 KM.


## 5. Compute district areas

In [24]:
districts = boundary.to_crs("EPSG:32736")

In [26]:
districts["area"] = districts["geometry"].area
districts

,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,...,lang,lang1,lang2,lang3,adm0_en,adm2_ref_n,center_lat,center_lon,geometry,area
0,Cidade de Lichinga,None,None,None,MZ0101,Niassa,None,None,None,MZ01,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((744887.147 8518331.261, 744839...",3.314671e+09
1,Cuamba,None,None,None,MZ0102,Niassa,None,None,None,MZ01,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((907909.129 8349537.545, 907909...",5.367498e+09
2,Lago,None,None,None,MZ0103,Niassa,None,None,None,MZ01,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((706578.95 8553188.17, 706525.5...",1.207040e+10
3,Chimbonila,None,None,None,MZ0104,Niassa,None,None,None,MZ01,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((750688.731 8481921.208, 750513...",3.828523e+09
4,Majune,None,None,None,MZ0105,Niassa,None,None,None,MZ01,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((832177.674 8473602.884, 832176...",1.170199e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,KaMaxaqueni,None,None,None,MZ1103,Cidade de Maputo,None,None,None,MZ11,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((457411.014 7129047.79, 457351....",1.265871e+07
157,KaMavota,None,None,None,MZ1104,Cidade de Maputo,None,None,None,MZ11,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((458729.313 7131843.623, 458558...",8.365025e+07
158,KaMubukwana,None,None,None,MZ1105,Cidade de Maputo,None,None,None,MZ11,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((455197.022 7131748.881, 455027...",5.802097e+07
159,KaTembe,None,None,None,MZ1106,Cidade de Maputo,None,None,None,MZ11,...,pt,None,None,None,None,None,NaN,NaN,"MULTIPOLYGON (((458271.344 7121818.733, 458529...",1.227505e+08


## 6. Spatial join — assign each road segment to its district

A `left` join with `predicate="intersects"` keeps every district (even ones with zero national road coverage) and attaches road segments that physically intersect each district polygon.

In [28]:
joineddistr = districts.sjoin(n_road_rep, how = "left", predicate = "intersects")

## 7. Aggregate: total national road length per district

In [30]:
results = joineddistr.groupby("adm2_name")["length"].sum()/1000
print(f"The result I found was that the district are covered in:\n{results}")

The result I found was that the district are covered in:
adm2_name
Alto Molocue    148.883007
Ancuabe         225.096053
Angoche          86.108704
Angónia          43.333447
Balama          135.292097
                   ...    
Vanduzi         121.967780
Vilankulo       209.507389
Xai-Xai          18.877601
Zavala           93.148049
Zumbo           212.640925
Name: length, Length: 161, dtype: float64


## 8. Bonus metric — road density

Raw length is biased toward large districts. Normalizing by area (km of road per km² of district) gives a fairer comparison of how *densely* each district is served by national roads.

In [ ]:
districts_results = districts.merge(
    results.rename("road_km"), on="adm2_name", how="left"
)
districts_results["road_km"] = districts_results["road_km"].fillna(0)
districts_results["area_km2"] = districts_results["area"] / 1e6
districts_results["road_density_km_per_km2"] = (
    districts_results["road_km"] / districts_results["area_km2"]
)
districts_results.sort_values("road_density_km_per_km2", ascending=False).head(10)


## 9. Visualization — overview map

Context map showing the district boundaries with the national road network overlaid.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 12))
districts.plot(ax=ax, color="#f2f2f2", edgecolor="#999999", linewidth=0.5)
n_road_rep.plot(ax=ax, color="#d73027", linewidth=1.2)
ax.set_title("Mozambique — National Road Network (N-Roads) by District", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("outputs/01_overview_map.png", dpi=200, bbox_inches="tight")
plt.show()


## 10. Visualization — choropleth of road coverage per district

This is the headline map: districts shaded by total national road length (km).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
districts_results.plot(
    column="road_km",
    cmap="YlOrRd",
    linewidth=0.5,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "National Road Length (km)", "shrink": 0.6},
    ax=ax,
)
ax.set_title("National Road Coverage per District (km)", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("outputs/02_choropleth_road_length.png", dpi=200, bbox_inches="tight")
plt.show()


## 11. Visualization — district ranking (bar chart)

A ranked bar chart is often easier to read precisely than a map — good as a companion figure.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
results.sort_values().plot(kind="barh", ax=ax, color="#2c7fb8")
ax.set_xlabel("National Road Length (km)")
ax.set_ylabel("District")
ax.set_title("Districts Ranked by National Road Coverage")
plt.tight_layout()
plt.savefig("outputs/03_district_ranking_bar.png", dpi=200, bbox_inches="tight")
plt.show()


## 12. Export results

In [32]:
results.to_csv("National_Road_Coverage_per_District.csv")#index=False)

In [ ]:
districts_results[["adm2_name", "road_km", "area_km2", "road_density_km_per_km2"]].to_csv(
    "outputs/district_road_density.csv", index=False
)


## Key takeaways

*(Fill in with your actual numbers once you rerun this notebook)*

- Total national road length analyzed: **___ km**
- District with the highest national road coverage: **___**
- District with the highest road *density* (km per km²): **___**
- Districts with zero national road coverage: **___**

## Possible extensions
- Break down by road surface type or road class (secondary/tertiary roads) for a fuller accessibility picture.
- Bring in population data to compute roads-per-capita, a classic accessibility indicator.
- Compare against health facility or market locations to estimate service-area accessibility.
